# 教程 05：表格类数据预测 — TabNet

**学习目标** ⭐
- 了解 TabNet 的原理和优势
- 使用本仓库的 TabNet 参考实现
- 掌握特征选择可视化和模型解释方法

---

## TabNet 简介

TabNet (Arik & Pfister, AAAI 2021) 是专为表格数据设计的深度学习模型，核心特点：

1. **序列注意力机制**：多步决策，每一步选择一部分特征
2. **Sparsemax 激活**：输出的注意力掩码具有稀疏性（部分特征被完全忽略）
3. **实例级特征选择**：每个样本可以选择不同的特征子集
4. **自监督预训练**：无需标签即可预训练，充分挖掘表格数据中的模式

> 传统方法（XGBoost、LightGBM）在表格数据上表现优秀，但不支持端到端的特征选择和可解释性。
> TabNet 通过注意力机制在决策过程中自动进行特征选择，同时保持深度学习的表达能力。

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, repo_root)

try:
    from prediction.TabularData.tabnet import (
        TabNetEncoder, TabNetClassifier, TabNetRegressor, Sparsemax
    )
    import torch
    import numpy as np
    print("TabNet 模块加载成功 ✅")
except ImportError as e:
    print(f"加载失败: {e}")

## 生成合成数据集

创建一个合成表格数据集：10 个特征中只有 4 个与目标相关，验证 TabNet 能否自动筛选出重要特征。

In [ ]:
import torch
import numpy as np

np.random.seed(42)
torch.manual_seed(42)

n_samples, n_features = 200, 10
X = np.random.randn(n_samples, n_features).astype(np.float32)

# 只有特征 2, 5, 7, 9 与目标相关
y_reg = (0.5 * X[:, 2] + 1.0 * X[:, 5] ** 2 + 0.3 * np.sin(X[:, 7]) + 0.8 * X[:, 9])
y_reg += np.random.randn(n_samples) * 0.1  # 添加噪声

X_t = torch.from_numpy(X)
y_t = torch.from_numpy(y_reg.astype(np.float32))

print(f"数据形状: X = {X_t.shape}, y = {y_t.shape}")
print(f"相关特征: 2, 5, 7, 9")

In [ ]:
model = TabNetRegressor(input_dim=n_features, dim=16, n_steps=3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

print("训练 TabNet 回归模型...")
for epoch in range(200):
    optimizer.zero_grad()
    pred, masks = model(X_t)
    loss = (pred - y_t).pow(2).mean()
    loss.backward()
    optimizer.step()

print(f"最终 MSE: {loss.item():.6f}")
print(f"RMSE: {torch.sqrt(loss).item():.4f}")

## 特征选择分析

TabNet 的注意力掩码直接展示了每个样本选择哪些特征，实现实例级可解释性。

In [ ]:
model.eval()
with torch.no_grad():
    pred, masks = model(X_t[:16])  # 取前 16 个样本

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for step, (mask, ax) in enumerate(zip(masks, axes)):
        im = ax.imshow(mask.numpy(), aspect='auto', cmap='YlOrRd', vmin=0, vmax=0.5)
        ax.set_xlabel('特征')
        ax.set_ylabel('样本')
        ax.set_title(f'决策步 {step + 1} 注意力掩码')
        plt.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.show()

    # 汇总特征重要性
    agg_importance = sum(mask.mean(dim=0) for mask in masks) / len(masks)
    print("\n特征重要性排序:")
    for idx in np.argsort(agg_importance.numpy())[::-1]:
        relevant = "★" if idx in [2, 5, 7, 9] else " "
        print(f"  特征 {idx}: {agg_importance[idx]:.4f} {relevant}")
except ImportError:
    agg_importance = sum(mask.mean(dim=0) for mask in masks) / len(masks)
    print("特征重要性:", agg_importance.numpy())

## 总结 📋

- TabNet 通过 Sparsemax 注意力实现实例级特征选择
- 在多步决策过程中，不同步骤关注不同的特征子集
- 可视化结果表明，TabNet 成功识别出与目标相关的特征（2, 5, 7, 9）
- 本仓库的 `prediction/TabularData/tabnet.py` 是完整的 PyTorch 参考实现

## 思考练习 ✏️

1. 增加无关特征的数量（如从 10 增加到 50），观察 TabNet 的特征筛选能力
2. 在真实数据集（如 UCI 仓库的房价预测）上训练 TabNet
3. 比较 TabNet 和 XGBoost 在该数据集上的表现差异